In [1]:
# ============================================================
# Cell 1: Environment + Project Path
# ============================================================

from pathlib import Path
import sys
import os
import math
import time

import torch


def find_project_root():
    """
    Locate the MyGPT2 project root.
    """

    candidates = []

    cwd = Path.cwd().resolve()

    candidates.append(cwd)
    candidates.extend(cwd.parents)

    try:
        notebook_dir = Path(
            os.environ.get("PWD", str(cwd))
        ).resolve()

        candidates.append(notebook_dir)
        candidates.extend(notebook_dir.parents)

    except Exception:
        pass

    for candidate in candidates:

        if (
            (candidate / "model").is_dir()
            and
            (candidate / "tokenizer").is_dir()
            and
            (candidate / "training").is_dir()
            and
            (candidate / "train.py").is_file()
        ):
            return candidate

    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find MyGPT2 project root.\n\n"
        "Expected:\n"
        "D:\\Gpt2_v01\\\n"
        "├── model\\\n"
        "├── tokenizer\\\n"
        "├── training\\\n"
        "└── train.py"
    )


PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

else:

    DEVICE = torch.device("cpu")


print("=" * 75)
print("MyGPT2 Training Evaluation")
print("=" * 75)

print(f"Current Directory : {Path.cwd()}")
print(f"Project Root      : {PROJECT_ROOT}")
print(f"PyTorch           : {torch.__version__}")
print(f"Device            : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU               : "
        f"{torch.cuda.get_device_name(0)}"
    )

print("=" * 75)

print()
print("Environment : ✅ PASSED")

MyGPT2 Training Evaluation
Current Directory : d:\Gpt2_v01\evaluation\notebooks
Project Root      : D:\Gpt2_v01
PyTorch           : 2.13.0+cu132
Device            : cuda
GPU               : NVIDIA GeForce RTX 5060 Ti

Environment : ✅ PASSED


In [2]:
# ============================================================
# Cell 2: MyGPT2 Imports
# ============================================================

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer

from training.checkpoint import load_checkpoint

print("=" * 75)
print("MyGPT2 Imports")
print("=" * 75)

print("GPTConfig       : ✅")
print("MyGPTModel      : ✅")
print("MyGPTTokenizer  : ✅")
print("load_checkpoint : ✅")

print("=" * 75)

MyGPT2 Imports
GPTConfig       : ✅
MyGPTModel      : ✅
MyGPTTokenizer  : ✅
load_checkpoint : ✅


In [3]:
# ============================================================
# Cell 3: Model / Tokenizer Paths
# ============================================================

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00010000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)


print("=" * 75)
print("Evaluation Files")
print("=" * 75)

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Tokenizer  : {TOKENIZER_PATH}")

if not CHECKPOINT_PATH.exists():

    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():

    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

print()
print("Checkpoint : ✅ FOUND")
print("Tokenizer  : ✅ FOUND")

Evaluation Files
Checkpoint : D:\Gpt2_v01\artifacts\checkpoints\final_step_00010000.pt
Tokenizer  : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json

Checkpoint : ✅ FOUND
Tokenizer  : ✅ FOUND


In [4]:
# ============================================================
# Cell 4: Model Configuration
# ============================================================

config = GPTConfig()


def find_config_value(config, names, default=None):

    for name in names:

        if hasattr(config, name):

            value = getattr(config, name)

            if value is not None:
                return value

    return default


VOCAB_SIZE = find_config_value(
    config,
    ["vocab_size"],
)

HIDDEN_SIZE = find_config_value(
    config,
    ["hidden_size", "n_embd", "d_model"],
)

NUM_LAYERS = find_config_value(
    config,
    ["num_layers", "n_layer"],
)

NUM_HEADS = find_config_value(
    config,
    [
        "num_heads",
        "num_attention_heads",
        "n_head",
    ],
)

INTERMEDIATE_SIZE = find_config_value(
    config,
    [
        "intermediate_size",
        "ffn_size",
        "n_inner",
    ],
)


# ------------------------------------------------------------
# Known training configuration
# ------------------------------------------------------------

SEQUENCE_LENGTH = 512

EXPECTED_VOCAB_SIZE = 32_000
EXPECTED_SEQUENCE_LENGTH = 512
EXPECTED_HIDDEN_SIZE = 768
EXPECTED_NUM_LAYERS = 12
EXPECTED_NUM_HEADS = 12
EXPECTED_INTERMEDIATE_SIZE = 3072


print("=" * 75)
print("Model Configuration")
print("=" * 75)

print(f"Vocabulary Size      : {VOCAB_SIZE:,}")
print(f"Sequence Length      : {SEQUENCE_LENGTH}")
print(f"Hidden Size          : {HIDDEN_SIZE}")
print(f"Transformer Layers   : {NUM_LAYERS}")
print(f"Attention Heads      : {NUM_HEADS}")
print(f"Intermediate Size    : {INTERMEDIATE_SIZE}")

print("=" * 75)


checks = {
    "Vocabulary Size": (
        VOCAB_SIZE,
        EXPECTED_VOCAB_SIZE,
    ),
    "Sequence Length": (
        SEQUENCE_LENGTH,
        EXPECTED_SEQUENCE_LENGTH,
    ),
    "Hidden Size": (
        HIDDEN_SIZE,
        EXPECTED_HIDDEN_SIZE,
    ),
    "Transformer Layers": (
        NUM_LAYERS,
        EXPECTED_NUM_LAYERS,
    ),
    "Attention Heads": (
        NUM_HEADS,
        EXPECTED_NUM_HEADS,
    ),
    "Intermediate Size": (
        INTERMEDIATE_SIZE,
        EXPECTED_INTERMEDIATE_SIZE,
    ),
}


print()
print("=" * 75)
print("Configuration Validation")
print("=" * 75)

for name, (actual, expected) in checks.items():

    if actual != expected:

        raise RuntimeError(
            f"{name} mismatch: "
            f"{actual} != {expected}"
        )

    print(
        f"{name:<25}: ✅ {actual}"
    )

print()
print("Configuration : ✅ PASSED")

Model Configuration
Vocabulary Size      : 32,000
Sequence Length      : 512
Hidden Size          : 768
Transformer Layers   : 12
Attention Heads      : 12
Intermediate Size    : 3072

Configuration Validation
Vocabulary Size          : ✅ 32000
Sequence Length          : ✅ 512
Hidden Size              : ✅ 768
Transformer Layers       : ✅ 12
Attention Heads          : ✅ 12
Intermediate Size        : ✅ 3072

Configuration : ✅ PASSED


In [5]:
# ============================================================
# Cell 5: Load Tokenizer + Model + Checkpoint
# ============================================================

print("=" * 75)
print("Loading Model and Checkpoint")
print("=" * 75)


tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)


if tokenizer.vocabulary_size != VOCAB_SIZE:

    raise RuntimeError(
        "Tokenizer vocabulary mismatch.\n"
        f"Tokenizer: {tokenizer.vocabulary_size}\n"
        f"Model:    {VOCAB_SIZE}"
    )


print(
    f"Tokenizer vocabulary : "
    f"{tokenizer.vocabulary_size:,}"
)


model = MyGPTModel(
    config
).to(DEVICE)


model.eval()


checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    scheduler=None,
    device=DEVICE,
    restore_rng=False,
)


print(
    f"Checkpoint version : "
    f"{checkpoint.get('checkpoint_version')}"
)

print(
    f"Checkpoint step    : "
    f"{checkpoint.get('global_step')}"
)

print(
    f"Training loss      : "
    f"{checkpoint.get('train_loss')}"
)

print()
print("Model + Checkpoint : ✅ LOADED")

Loading Model and Checkpoint
Tokenizer vocabulary : 32,000
Checkpoint version : 1.2
Checkpoint step    : 10000
Training loss      : 1.3898952007293701

Model + Checkpoint : ✅ LOADED


In [6]:
# ============================================================
# Cell 6: Load TinyStories Training Dataset
# ============================================================

from datasets import load_dataset


print("=" * 75)
print("Loading TinyStories Training Dataset")
print("=" * 75)


dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
)


print(
    f"Dataset : roneneldan/TinyStories"
)

print(
    f"Split   : train"
)

print(
    f"Documents available : "
    f"{len(dataset):,}"
)

print()

if "text" not in dataset.column_names:

    raise RuntimeError(
        "TinyStories dataset does not contain "
        "the expected 'text' column."
    )


print("Dataset loading : ✅ PASSED")

d:\Gpt2_v01\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TinyStories Training Dataset


Dataset : roneneldan/TinyStories
Split   : train
Documents available : 2,119,719

Dataset loading : ✅ PASSED


In [7]:
# ============================================================
# Cell 7: Dataset Inspection
# ============================================================

print("=" * 75)
print("Training Dataset Inspection")
print("=" * 75)

print(
    "Columns:"
)

print(
    dataset.column_names
)

print()

print(
    "First sample:"
)

print(
    dataset[0]
)

print()

sample_text = dataset[0]["text"]

print(
    "First sample preview:"
)

print(
    sample_text[:1000]
)

print()

print("Dataset inspection : ✅ PASSED")

Training Dataset Inspection
Columns:
['text']

First sample:
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

First sample preview:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button

In [8]:
# ============================================================
# Cell 8: Tokenizer Test
# ============================================================

test_text = dataset[0]["text"]

encoded = tokenizer.encode(
    test_text
)

if hasattr(encoded, "ids"):

    token_ids = encoded.ids

else:

    token_ids = encoded


print("=" * 75)
print("Tokenizer Test")
print("=" * 75)

print(
    f"Characters : {len(test_text)}"
)

print(
    f"Tokens     : {len(token_ids)}"
)

print(
    f"First tokens: {token_ids[:20]}"
)

if len(token_ids) == 0:

    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )

print()
print("Tokenizer test : ✅ PASSED")

Tokenizer Test
Characters : 701
Tokens     : 163
First tokens: [2, 740, 445, 15, 219, 546, 691, 822, 513, 769, 219, 10984, 251, 341, 1217, 17, 382, 1236, 286, 277]

Tokenizer test : ✅ PASSED


In [9]:
# ============================================================
# Cell 9: Build Training Token Stream
# ============================================================

MAX_EVALUATION_TOKENS = 500_000


print("=" * 75)
print("Building Training Token Stream")
print("=" * 75)

training_tokens = []

documents_processed = 0
documents_skipped = 0

tokens_collected = 0


for document in dataset:

    text = document.get("text", "")

    if not isinstance(text, str):
        documents_skipped += 1
        continue

    if not text.strip():
        documents_skipped += 1
        continue


    encoded = tokenizer.encode(text)

    if hasattr(encoded, "ids"):

        ids = encoded.ids

    else:

        ids = encoded


    if not ids:

        documents_skipped += 1
        continue


    remaining = (
        MAX_EVALUATION_TOKENS
        - tokens_collected
    )


    if len(ids) > remaining:

        ids = ids[:remaining]


    training_tokens.extend(ids)

    tokens_collected += len(ids)

    documents_processed += 1


    if tokens_collected >= MAX_EVALUATION_TOKENS:

        break


print(
    f"Documents processed : "
    f"{documents_processed:,}"
)

print(
    f"Documents skipped   : "
    f"{documents_skipped:,}"
)

print(
    f"Tokens collected    : "
    f"{tokens_collected:,}"
)


if tokens_collected < SEQUENCE_LENGTH + 1:

    raise RuntimeError(
        "Not enough tokens to create "
        "validation sequences."
    )


print()
print("Token stream : ✅ CREATED")

Building Training Token Stream
Documents processed : 2,299
Documents skipped   : 0
Tokens collected    : 500,000

Token stream : ✅ CREATED


In [10]:
# ============================================================
# Cell 10: Build Training Evaluation Sequences
# ============================================================

print("=" * 75)
print("Building Training Evaluation Sequences")
print("=" * 75)


usable_tokens = (
    len(training_tokens)
    - 1
)


num_sequences = (
    usable_tokens
    // SEQUENCE_LENGTH
)


evaluation_tokens = (
    num_sequences
    * SEQUENCE_LENGTH
)


inputs = []
labels = []


for i in range(num_sequences):

    start = (
        i * SEQUENCE_LENGTH
    )

    end = (
        start + SEQUENCE_LENGTH
    )


    input_sequence = training_tokens[
        start:end
    ]

    label_sequence = training_tokens[
        start + 1:end + 1
    ]


    if (
        len(input_sequence)
        != SEQUENCE_LENGTH
    ):
        continue

    if (
        len(label_sequence)
        != SEQUENCE_LENGTH
    ):
        continue


    inputs.append(
        input_sequence
    )

    labels.append(
        label_sequence
    )


print(
    f"Sequence Length      : "
    f"{SEQUENCE_LENGTH}"
)

print(
    f"Training sequences    : "
    f"{len(inputs):,}"
)

print(
    f"Evaluation tokens     : "
    f"{len(inputs) * SEQUENCE_LENGTH:,}"
)


if len(inputs) == 0:

    raise RuntimeError(
        "No training evaluation sequences "
        "were created."
    )


print()
print("Sequence generation : ✅ PASSED")

Building Training Evaluation Sequences
Sequence Length      : 512
Training sequences    : 976
Evaluation tokens     : 499,712

Sequence generation : ✅ PASSED


In [11]:
# ============================================================
# Cell 11: Training Evaluation DataLoader
# ============================================================

from torch.utils.data import TensorDataset, DataLoader


input_tensor = torch.tensor(
    inputs,
    dtype=torch.long,
)

label_tensor = torch.tensor(
    labels,
    dtype=torch.long,
)


evaluation_dataset = TensorDataset(
    input_tensor,
    label_tensor,
)


BATCH_SIZE = 8


evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


print("=" * 75)
print("Training Evaluation DataLoader")
print("=" * 75)

print(
    f"Batch Size       : {BATCH_SIZE}"
)

print(
    f"Number of batches: "
    f"{len(evaluation_loader):,}"
)

print()
print("DataLoader : ✅ CREATED")

Training Evaluation DataLoader
Batch Size       : 8
Number of batches: 122

DataLoader : ✅ CREATED


In [12]:
# ============================================================
# Cell 12: Training Sequence Inspection
# ============================================================

first_inputs, first_labels = next(
    iter(evaluation_loader)
)


print("=" * 75)
print("Training Evaluation Sequence Inspection")
print("=" * 75)

print(
    f"Input shape  : "
    f"{tuple(first_inputs.shape)}"
)

print(
    f"Label shape  : "
    f"{tuple(first_labels.shape)}"
)

print(
    f"Input dtype  : "
    f"{first_inputs.dtype}"
)

print(
    f"Label dtype  : "
    f"{first_labels.dtype}"
)

print()

print(
    "First 20 input tokens:"
)

print(
    first_inputs[0][:20].tolist()
)

print()

print(
    "First 20 target tokens:"
)

print(
    first_labels[0][:20].tolist()
)


# Verify next-token relationship

if not torch.equal(
    first_inputs[0][1:20],
    first_labels[0][:19],
):

    raise RuntimeError(
        "Input/label shifting is incorrect."
    )


print()
print("Sequence structure : ✅ PASSED")

Training Evaluation Sequence Inspection
Input shape  : (8, 512)
Label shape  : (8, 512)
Input dtype  : torch.int64
Label dtype  : torch.int64

First 20 input tokens:
[2, 740, 445, 15, 219, 546, 691, 822, 513, 769, 219, 10984, 251, 341, 1217, 17, 382, 1236, 286, 277]

First 20 target tokens:
[740, 445, 15, 219, 546, 691, 822, 513, 769, 219, 10984, 251, 341, 1217, 17, 382, 1236, 286, 277, 2502]

Sequence structure : ✅ PASSED


In [13]:
# ============================================================
# Cell 13: Run Training Evaluation
# ============================================================

print("=" * 75)
print("Running Training Evaluation")
print("=" * 75)


model.eval()


total_loss = 0.0
total_tokens = 0

top1_correct = 0
top5_correct = 0
top10_correct = 0


start_time = time.time()


with torch.no_grad():

    for batch_index, (
        input_ids,
        labels,
    ) in enumerate(evaluation_loader, start=1):


        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )


        output = model(
            input_ids=input_ids,
            labels=labels,
        )


        if isinstance(output, tuple):

            if len(output) != 2:

                raise RuntimeError(
                    "Expected model output "
                    "format: (logits, loss)."
                )

            logits, loss = output

        else:

            if not hasattr(output, "loss"):

                raise RuntimeError(
                    "Model output does not "
                    "contain loss."
                )

            logits = output.logits
            loss = output.loss


        if loss is None:

            raise RuntimeError(
                "Model returned None loss."
            )


        if not torch.isfinite(loss):

            raise RuntimeError(
                "Loss became NaN or infinite."
            )


        batch_token_count = (
            labels.numel()
        )


        total_loss += (
            loss.item()
            * batch_token_count
        )

        total_tokens += (
            batch_token_count
        )


        # ----------------------------------------------------
        # Top-K predictions
        # ----------------------------------------------------

        predictions = logits.argmax(
            dim=-1
        )


        top1_correct += (
            (predictions == labels)
            .sum()
            .item()
        )


        topk_values = torch.topk(
            logits,
            k=10,
            dim=-1,
        ).indices


        expanded_labels = (
            labels.unsqueeze(-1)
        )


        top5_correct += (
            (
                topk_values[:, :, :5]
                == expanded_labels
            )
            .any(dim=-1)
            .sum()
            .item()
        )


        top10_correct += (
            (
                topk_values
                == expanded_labels
            )
            .any(dim=-1)
            .sum()
            .item()
        )


        if (
            batch_index == 1
            or batch_index % 10 == 0
        ):

            batch_accuracy = (
                top1_correct
                /
                total_tokens
                * 100
            )

            print(
                f"Batch {batch_index:>5} | "
                f"Loss {loss.item():.6f} | "
                f"Accuracy "
                f"{batch_accuracy:.2f}%"
            )


elapsed_time = (
    time.time()
    - start_time
)


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

training_loss = (
    total_loss
    /
    total_tokens
)


training_perplexity = math.exp(
    min(training_loss, 20)
)


top1_accuracy = (
    top1_correct
    /
    total_tokens
    * 100
)


top5_accuracy = (
    top5_correct
    /
    total_tokens
    * 100
)


top10_accuracy = (
    top10_correct
    /
    total_tokens
    * 100
)


print()
print("=" * 75)
print("Training Evaluation Completed")
print("=" * 75)

print(
    f"Training Loss       : "
    f"{training_loss:.6f}"
)

print(
    f"Token Accuracy      : "
    f"{top1_accuracy:.4f}%"
)

print(
    f"Top-5 Accuracy      : "
    f"{top5_accuracy:.4f}%"
)

print(
    f"Top-10 Accuracy     : "
    f"{top10_accuracy:.4f}%"
)

print(
    f"Tokens Evaluated    : "
    f"{total_tokens:,}"
)

print(
    f"Validation Batches  : "
    f"{len(evaluation_loader):,}"
)

print(
    f"Evaluation Time     : "
    f"{elapsed_time:.2f}s"
)

print("=" * 75)

Running Training Evaluation
Batch     1 | Loss 3.070619 | Accuracy 42.31%
Batch    10 | Loss 4.018770 | Accuracy 35.64%
Batch    20 | Loss 4.102195 | Accuracy 34.84%
Batch    30 | Loss 4.082289 | Accuracy 34.40%
Batch    40 | Loss 1.623901 | Accuracy 39.80%
Batch    50 | Loss 1.652323 | Accuracy 42.27%
Batch    60 | Loss 4.003308 | Accuracy 44.05%
Batch    70 | Loss 4.262300 | Accuracy 42.69%
Batch    80 | Loss 4.155706 | Accuracy 42.36%
Batch    90 | Loss 4.386428 | Accuracy 41.42%
Batch   100 | Loss 3.964903 | Accuracy 40.72%
Batch   110 | Loss 1.907831 | Accuracy 41.04%
Batch   120 | Loss 1.869656 | Accuracy 42.15%

Training Evaluation Completed
Training Loss       : 3.306426
Token Accuracy      : 42.4126%
Top-5 Accuracy      : 67.4611%
Top-10 Accuracy     : 74.6578%
Tokens Evaluated    : 499,712
Validation Batches  : 122
Evaluation Time     : 18.69s


In [14]:
# ============================================================
# Cell 14: Training Loss Comparison
# ============================================================

saved_training_loss = checkpoint.get(
    "train_loss"
)


print("=" * 75)
print("Checkpoint Training Loss Comparison")
print("=" * 75)

print(
    f"Checkpoint saved loss : "
    f"{saved_training_loss:.6f}"
)

print(
    f"Evaluation loss       : "
    f"{training_loss:.6f}"
)


loss_difference = (
    training_loss
    - saved_training_loss
)


print(
    f"Loss difference       : "
    f"{loss_difference:+.6f}"
)


if abs(loss_difference) < 0.10:

    print()
    print(
        "Training evaluation matches "
        "checkpoint training loss reasonably well."
    )

    print(
        "Pipeline consistency : ✅ GOOD"
    )

else:

    print()
    print(
        "WARNING: Training evaluation differs "
        "substantially from the checkpoint loss."
    )

    print(
        "Pipeline consistency : ⚠️ INVESTIGATE"
    )


print("=" * 75)

Checkpoint Training Loss Comparison
Checkpoint saved loss : 1.389895
Evaluation loss       : 3.306426
Loss difference       : +1.916531

Pipeline consistency : ⚠️ INVESTIGATE


In [15]:
# ============================================================
# Cell 15: Training vs Validation Comparison
# ============================================================

VALIDATION_LOSS = 3.418222
VALIDATION_PERPLEXITY = 30.5151

VALIDATION_TOP1 = 40.8577
VALIDATION_TOP5 = 66.2470
VALIDATION_TOP10 = 73.7269


print("=" * 75)
print("Training vs Validation")
print("=" * 75)


print(
    f"Training Loss        : "
    f"{training_loss:.6f}"
)

print(
    f"Validation Loss      : "
    f"{VALIDATION_LOSS:.6f}"
)


loss_gap = (
    VALIDATION_LOSS
    - training_loss
)


print(
    f"Loss Gap             : "
    f"{loss_gap:+.6f}"
)


print()

print(
    f"Training Perplexity  : "
    f"{training_perplexity:.4f}"
)

print(
    f"Validation Perplexity: "
    f"{VALIDATION_PERPLEXITY:.4f}"
)


print()

print(
    f"Training Top-1       : "
    f"{top1_accuracy:.4f}%"
)

print(
    f"Validation Top-1     : "
    f"{VALIDATION_TOP1:.4f}%"
)

print()

print(
    f"Training Top-5       : "
    f"{top5_accuracy:.4f}%"
)

print(
    f"Validation Top-5     : "
    f"{VALIDATION_TOP5:.4f}%"
)

print()

print(
    f"Training Top-10      : "
    f"{top10_accuracy:.4f}%"
)

print(
    f"Validation Top-10    : "
    f"{VALIDATION_TOP10:.4f}%"
)


print("=" * 75)

Training vs Validation
Training Loss        : 3.306426
Validation Loss      : 3.418222
Loss Gap             : +0.111796

Training Perplexity  : 27.2874
Validation Perplexity: 30.5151

Training Top-1       : 42.4126%
Validation Top-1     : 40.8577%

Training Top-5       : 67.4611%
Validation Top-5     : 66.2470%

Training Top-10      : 74.6578%
Validation Top-10    : 73.7269%


In [16]:
# ============================================================
# Cell 16: Generalization Diagnosis
# ============================================================

print("=" * 75)
print("Generalization Diagnosis")
print("=" * 75)


relative_loss_gap = (
    VALIDATION_LOSS
    /
    training_loss
)


print(
    f"Validation / Training Loss Ratio : "
    f"{relative_loss_gap:.2f}x"
)


print(
    f"Absolute Loss Gap                : "
    f"{loss_gap:+.6f}"
)


if relative_loss_gap >= 2.0:

    diagnosis = (
        "SIGNIFICANT GENERALIZATION GAP"
    )

elif relative_loss_gap >= 1.5:

    diagnosis = (
        "MODERATE GENERALIZATION GAP"
    )

elif relative_loss_gap >= 1.2:

    diagnosis = (
        "SMALL GENERALIZATION GAP"
    )

else:

    diagnosis = (
        "GOOD GENERALIZATION"
    )


print()
print(
    f"Diagnosis : {diagnosis}"
)


print("=" * 75)

Generalization Diagnosis
Validation / Training Loss Ratio : 1.03x
Absolute Loss Gap                : +0.111796

Diagnosis : GOOD GENERALIZATION


In [ ]:
# ============================================================
# Cell 17: Final Evaluation Report
# ============================================================

print()
print("=" * 75)
print("MyGPT2 Training Evaluation Summary")
print("=" * 75)

print(
    f"Checkpoint Step       : "
    f"{checkpoint.get('global_step'):,}"
)

print(
    f"Model Parameters      : "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    f"Training Documents    : "
    f"{documents_processed:,}"
)

print(
    f"Training Sequences    : "
    f"{len(inputs):,}"
)

print(
    f"Tokens Evaluated      : "
    f"{total_tokens:,}"
)

print()

print(
    f"Checkpoint Loss       : "
    f"{saved_training_loss:.6f}"
)

print(
    f"Evaluation Loss       : "
    f"{training_loss:.6f}"
)

print(
    f"Validation Loss       : "
    f"{VALIDATION_LOSS:.6f}"
)

print()

print(
    f"Training Perplexity   : "
    f"{training_perplexity:.4f}"
)

print(
    f"Validation Perplexity : "
    f"{VALIDATION_PERPLEXITY:.4f}"
)

print()

print(
    f"Training Top-1        : "
    f"{top1_accuracy:.4f}%"
)

print(
    f"Validation Top-1      : "
    f"{VALIDATION_TOP1:.4f}%"
)

print()

print(
    f"Training Top-5        : "
    f"{top5_accuracy:.4f}%"
)

print(
    f"Validation Top-5      : "
    f"{VALIDATION_TOP5:.4f}%"
)

print()

print(
    f"Training Top-10       : "
    f"{top10_accuracy:.4f}%"
)

print(
    f"Validation Top-10     : "
    f"{VALIDATION_TOP10:.4f}%"
)

print()

print(
    f"Generalization        : "
    f"{diagnosis}"
)

print()

print(
    "Training Evaluation   : ✅ COMPLETED"
)

print("=" * 75)